# Notebook 03 — Job Code Spine + MOS Risk Overlay

Build the authoritative **Job Code spine** from the VA Duty MOS Noise Exposure Listing (Fast Letter 10-35), then layer the hand-curated `data/mos_risk.yaml` overlay on top.

**Spine** — every Job Code across every branch (Army/USMC MOS, Navy/Coast Guard Rating, Air Force AFSC) gets one `:CFR:JobCode {code, title, branch, type}` node and one `:NOISE_EXPOSURE {probability}` edge to the `hearing` Anatomy. Sourced from the VA — legally authoritative.

**Overlay** — hand-curated musculoskeletal-risk inferences from publicly available duty descriptions. Attaches `:RISK_FOR {confidence, source}` edges to existing `:Anatomy` and `:Symptom` nodes. By construction it cannot invent JobCodes.

**Prereqs**

- `make up` — Neo4j running
- `data/duty_mos_noise.xlsx` committed (covered by `data/SOURCE.md`)

## 1. Locate the xlsx and inspect its column structure

Each branch lives on its own sheet, with slightly different columns. The ingester has a per-sheet parser dispatch.

In [ ]:
from pathlib import Path
from openpyxl import load_workbook

xlsx_path = Path('..') / 'data' / 'duty_mos_noise.xlsx'
wb = load_workbook(xlsx_path, data_only=True)
for name in wb.sheetnames:
    ws = wb[name]
    header = next(ws.iter_rows(values_only=True), None)
    print(f'{name:<22} {ws.max_row:>4} rows  header={header}')

## 2. Ingest the spine

MERGE one JobCode per row plus the NOISE_EXPOSURE edge to the `hearing` Anatomy. Unparseable rows go to `data/review_queue.jsonl`.

In [ ]:
from va_agent.graph.driver import GraphDriver
from va_agent.ingestion.jobcodes import ingest_job_code_spine

driver = GraphDriver.from_env()
report = ingest_job_code_spine(xlsx_path, driver)

print(f'rows seen:     {report.rows_seen}')
print(f'rows written:  {report.rows_written}')
print(f'rows rejected: {len(report.rows_rejected)}')
print()
print('Per-branch counts:')
for branch, n in sorted(report.per_branch.items(), key=lambda kv: -kv[1]):
    print(f'  {branch:<16} {n:>5}')

## 3. Probability distribution across the spine

In [ ]:
rows = driver.cfr_read(
    '''MATCH (jc:CFR:JobCode)-[r:NOISE_EXPOSURE]->(:CFR:Anatomy {name: 'hearing'})
       RETURN jc.branch AS branch, r.probability AS probability, count(*) AS n
       ORDER BY branch, probability'''
)
for r in rows:
    print(f"  {r['branch']:<16} {r['probability']:<18} {r['n']:>5}")

## 4. Apply the curated risk overlay

Loads `data/mos_risk.yaml`. For each entry, MATCHes the JobCode in the spine and MERGEs `:RISK_FOR` edges to existing anatomy/symptom nodes. Refuses to invent JobCodes — if the overlay references a code the spine doesn't have, it raises `OverlayError`.

In [ ]:
from va_agent.ingestion.risk_overlay import apply_mos_risk_overlay

overlay_path = Path('..') / 'data' / 'mos_risk.yaml'
overlay_report = apply_mos_risk_overlay(overlay_path, driver)

print(f'entries seen:     {overlay_report.entries_seen}')
print(f'entries applied:  {overlay_report.entries_applied}')
print(f'risk edges:       {overlay_report.risk_edges_written}')

## 5. Demo query — what does the graph predict for Army MOS 15T (UH-60 Crew Chief)?

Both the authoritative noise-exposure edge from the spine *and* the curated musculoskeletal risk edges from the overlay surface together. Notice the `confidence` field — the overlay edges are tagged `medium` because they're inferences from duty descriptions, not legally authoritative VA determinations.

In [ ]:
rows = driver.cfr_read(
    '''MATCH (jc:CFR:JobCode {code: '15T', branch: 'Army'})-[r]->(t)
       RETURN type(r) AS edge, labels(t) AS target_labels,
              coalesce(t.name, t.title) AS target,
              r.probability AS noise_prob,
              r.confidence  AS confidence,
              r.source      AS source
       ORDER BY edge, target'''
)
for r in rows:
    label = ':'.join(L for L in r['target_labels'] if L != 'CFR')
    if r['edge'] == 'NOISE_EXPOSURE':
        print(f"  NOISE_EXPOSURE  -> ({label} {r['target']!r})  prob={r['noise_prob']}")
    else:
        print(f"  {r['edge']:<14} -> ({label} {r['target']!r})  confidence={r['confidence']}  source={r['source']}")

## 6. Which JobCodes got RISK_FOR edges from the overlay?

In [ ]:
rows = driver.cfr_read(
    '''MATCH (jc:CFR:JobCode)-[r:RISK_FOR]->(t)
       RETURN jc.branch AS branch, jc.code AS code, jc.title AS title,
              count(r) AS n_risk_edges
       ORDER BY branch, code'''
)
for r in rows:
    print(f"  {r['branch']:<14} {r['code']:<10} {r['title']:<40} {r['n_risk_edges']} edges")

driver.close()